# 16. statsmodels — `statsmodels`
**Python Libraries Lab · Phase 1 · Data Science**

_Regression and statistical models with full, readable reports._

✅ **Runs in Colab** as-is.

statsmodels is for explaining, not just predicting. It fits classic statistical models, like linear regression, and gives you a full report. How much does each factor matter, how sure are we, and how well does the model fit. Economists, researchers and analysts use it when the why matters as much as the number.

**In this notebook:** You will create ShopEasy's weekly marketing data, fit a regression with a simple formula like sales tilde ads plus discount, read the summary table, make a prediction, and look at the trend in a time series.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q statsmodels
import importlib.metadata as md
for pkg in ['statsmodels']:
    print(pkg, md.version(pkg))

## Step 1 · Weekly marketing data
104 weeks of ad spend, discount and sales. We know the true effect (3 per rupee of ads), so we can check the model finds it.

In [ ]:
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

rng = np.random.default_rng(42)
weeks = 104
ads = rng.uniform(5_000, 50_000, weeks)
discount = rng.choice([0, 5, 10, 15, 20], weeks)
festival = (rng.random(weeks) < 0.15).astype(int)
sales = 40_000 + 3 * ads + 2_500 * discount + 30_000 * festival + rng.normal(0, 12_000, weeks)
mkt = pd.DataFrame({"sales": sales.round(), "ads": ads.round(), "discount": discount, "festival": festival})
mkt.head()

## Step 2 · Fit a regression with a formula
`'sales ~ ads + discount + festival'` reads as: explain sales using these three.

In [ ]:
model = smf.ols("sales ~ ads + discount + festival", data=mkt).fit()
print(model.summary())

## Step 3 · Pull out the key numbers
The parts of the report you will use most.

In [ ]:
print("Coefficients:\n", model.params.round(2))
print("\np-values:\n", model.pvalues.round(4))
print(f"\nR-squared: {model.rsquared:.3f}")
print("95% confidence interval for ads:", model.conf_int().loc["ads"].round(2).tolist())

## Step 4 · Predict a plan
What sales should we expect with 30,000 in ads, a 10% discount, in a festival week?

In [ ]:
plan = pd.DataFrame({"ads": [30_000, 30_000], "discount": [10, 10], "festival": [0, 1]})
pred = model.get_prediction(plan).summary_frame(alpha=0.05)
pred[["mean", "obs_ci_lower", "obs_ci_upper"]].round(0)

## Step 5 · Time series: trend and season
`seasonal_decompose` splits a series into trend, a repeating weekly pattern, and noise.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

days = pd.date_range("2026-01-01", periods=120, freq="D")
weekly_pattern = np.tile([0, 0, 0, 0, 500, 1500, 1200], 18)[:120]
daily = pd.Series(5000 + np.arange(120) * 15 + weekly_pattern + rng.normal(0, 300, 120), index=days)

parts = seasonal_decompose(daily, period=7)
print("Trend grew from", round(parts.trend.dropna().iloc[0]), "to", round(parts.trend.dropna().iloc[-1]))
print("Weekly pattern (Mon..Sun):", parts.seasonal.iloc[:7].round(0).tolist())

## ✍️ Your turn
Fit `smf.ols('sales ~ ads', data=mkt)` (ads only) and store its **R-squared** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    expected = smf.ols("sales ~ ads", data=mkt).fit().rsquared
    ok = answer is not None and abs(float(answer) - expected) < 1e-9
    return ok, f"Expected about {expected:.3f}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = smf.ols("sales ~ ads", data=mkt).fit().rsquared

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Regression**
- **Coefficient**
- **R-squared**
- **p-value**
- **Formula API**
- **Time series**

**Next:** 17. Symbolic Maths (`sympy`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/17_sympy.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install statsmodels
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.